<a href="https://colab.research.google.com/github/willy410-hub/extraction-sft-rl/blob/main/notebooks/try_extraction_sft_rl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Try extraction-sft-rl: input, model, output

This notebook runs a small open model (Qwen2.5-0.5B-Instruct) on messy, invoice-like messages and checks its JSON answer against the correct one.

All data is synthetic. Project: https://github.com/willy410-hub/extraction-sft-rl

**Faster (optional):** Runtime, then Change runtime type, then pick **T4 GPU**. It also works on CPU, just slower.

Run the cells from top to bottom (Shift + Enter).

## 1. Download the project and install what it needs
If Colab asks you to restart the session after this cell, restart it and continue with step 2.

In [1]:
!git clone -q https://github.com/willy410-hub/extraction-sft-rl
!pip install -q "transformers==5.21.0" "trl==1.16.0" "peft==0.21.2" "datasets==5.1.0" "accelerate==1.15.0"

fatal: destination path 'extraction-sft-rl' already exists and is not an empty directory.


## 1b. Run the tests (no model download needed)

In [2]:
!cd /content/extraction-sft-rl && python -m pytest -q

.............................................................            [100%]
61 passed in 20.60s


## 2. Look at the data
Each example is a messy message plus the correct answer. The answer is chosen first and then written into the text, so we always know what is right.

In [3]:
%cd /content/extraction-sft-rl
from extraction_rl.generator import make_split

examples = make_split("test", 4)   # change 4 to see more
for e in examples:
    print("NOISE TYPE:", e["noise"])
    print(e["text"])
    print("CORRECT ANSWER:", e["truth"])
    print("-" * 60)

/content/extraction-sft-rl
NOISE TYPE: clean
Hi team,
Please process the following.
Amount: $2,205.45
Customer name: أحمد سمير
Invoice: INV-73465
Customer ID: C-6475
Date: 2024-04-01
Thanks
CORRECT ANSWER: {'invoice_id': 'INV-73465', 'date': '2024-04-01', 'amount': 2205.45, 'currency': 'USD', 'customer_id': 'C-6475', 'customer_name': 'أحمد سمير'}
------------------------------------------------------------
NOISE TYPE: encoding
Hi team,
Please process the following.
Amount: 3.480,36 GBP
Invoice: INV-94665
Date: 19/10/2025
Customer name: BjÃ¶rn Ã…strÃ¶m
Customer ID: C-5191
Thanks
CORRECT ANSWER: {'invoice_id': 'INV-94665', 'date': '2025-10-19', 'amount': 3480.36, 'currency': 'GBP', 'customer_id': 'C-5191', 'customer_name': 'Björn Åström'}
------------------------------------------------------------
NOISE TYPE: missing_field
Amount 2572.61 GBP, Date 2026-01-20, Invoice INV-86784, Customer name Maria Rossi
CORRECT ANSWER: {'invoice_id': 'INV-86784', 'date': '2026-01-20', 'amount': 2572.61,

## 3. Run the model and grade its answers
The first run downloads the model (about 1 GB). Score 1.0 means all 6 fields are right.
**Note:** this is the untuned base model, before any training. On the 400-item test split it gets 0.345 exact match; the SFT model in the repo gets 0.97 to 0.995. The trained adapter is not published yet, so mistakes below are expected.

In [4]:
from extraction_rl.inference import load_model, generate
from extraction_rl.verifier import verify

model, tok = load_model()
outputs, _ = generate(model, tok, [e["text"] for e in examples])

for e, out in zip(examples, outputs):
    result = verify(out, e["truth"])
    print("INPUT:\n" + e["text"])
    print("MODEL OUTPUT:", out)
    print("CORRECT ANSWER:", e["truth"])
    print("SCORE:", result["score"], "| fields right:", result["per_field"])
    print("-" * 60)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

INPUT:
Hi team,
Please process the following.
Amount: $2,205.45
Customer name: أحمد سمير
Invoice: INV-73465
Customer ID: C-6475
Date: 2024-04-01
Thanks
MODEL OUTPUT: {"invoice_id": "INV-73465", "date": "2024-04-01", "amount": 2205.45, "currency": "EUR", "customer_id": "C-6475", "customer_name": "أحمد سمير"}
CORRECT ANSWER: {'invoice_id': 'INV-73465', 'date': '2024-04-01', 'amount': 2205.45, 'currency': 'USD', 'customer_id': 'C-6475', 'customer_name': 'أحمد سمير'}
SCORE: 0.8333333333333334 | fields right: {'invoice_id': True, 'date': True, 'amount': True, 'currency': False, 'customer_id': True, 'customer_name': True}
------------------------------------------------------------
INPUT:
Hi team,
Please process the following.
Amount: 3.480,36 GBP
Invoice: INV-94665
Date: 19/10/2025
Customer name: BjÃ¶rn Ã…strÃ¶m
Customer ID: C-5191
Thanks
MODEL OUTPUT: {"invoice_id": "INV-94665", "date": "2025-10-19", "amount": 3.48036, "currency": "GBP", "customer_id": "C-5191", "customer_name": "Björn Åst

## 4. Try your own message
Write any invoice-like text below. There is no correct answer here, so you just see what the model returns.

In [5]:
my_text = """Hello, please pay invoice INV-12345 for customer Ana Peña (ID C-2024),
total 1,250.00 EUR, dated 3 March 2026."""

out, _ = generate(model, tok, [my_text])
print(out[0])

{"invoice_id": "INV-12345", "date": "2026-03-03", "amount": 1250.0, "currency": "EUR", "customer_id": "C-2024", "customer_name": "Ana Peña"}
